# BFX / 00 — Bond–FX data: coverage, co-movement by hour, hedge ratios through time

Protocol: [`BFX 001`](../../discussions/Bond_FX_Pairs/001_claude_bfx01_preregistration.md). Descriptive only, nothing is selected here. Uses **H1** (2018–2021) so the second half stays unseen until its own notebooks.

- Bars per pairing and year (5-minute bars where every leg was quoted).
- Correlation of 5-minute FX returns with each bond's returns, by UTC hour: when the link is strong.
- Rolling 60-day hedge ratios (no-intercept regression of FX returns on bond returns), to see whether the link is stable.

In [ ]:
WINDOW = "H1"

import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("BFX_SYNTHETIC") == "1"
if SYNTHETIC:                                   # never touch the real market-data folder in a smoke run
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("BFX_SYNTH_DIR") or tempfile.mkdtemp(prefix="bfx_")
    os.environ["BFX_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from equity_trading import bond_seasonality as B
from equity_trading import bond_fx as X

cfg = X.BFXConfig()
if SYNTHETIC:
    cfg = replace(cfg, n_placebo=2, n_boot=2_000)
PAIRS = list(X.PAIRINGS)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)
OUT = Path.cwd() / ("00_synthetic" if SYNTHETIC else "00"); OUT.mkdir(exist_ok=True)
START, END = X.WINDOWS[WINDOW]

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    if not (base / "EURUSD").exists():
        X.write_synthetic_market(base, dt.date(2017, 12, 28), dt.date(2025, 12, 31), seed=5, spike="2019-05-15 10:00")

## 1. Bars

In [ ]:
legs = {}
frames = {p: X.frame(p, WINDOW, cfg, legs) for p in PAIRS}
print(f"window {WINDOW} {START} -> {END}")
display(pd.DataFrame([{"pairing": p, "bonds": " + ".join(X.BOND_SHORT[b] for b in fr.bonds), "bars": fr.n,
                       "days": len(np.unique(fr.day)), "bars per day (median)": fr.bars_per_day,
                       "counted years": ", ".join(map(str, X.counted_years(fr, cfg)))} for p, fr in frames.items()]))
display(pd.DataFrame([{"leg": s, "file share": round(l.file_share, 4), "missing days": len(l.missing),
                       "spike minutes": len(l.spikes), "contract switches": int((l.rolls["method"] != "extra").sum()) if len(l.rolls) else 0,
                       "switch methods": l.rolls.loc[l.rolls["method"] != "extra", "method"].value_counts().to_dict() if len(l.rolls) else {}}
                      for s, l in legs.items()]))
assert all(l.file_share >= cfg.min_file_share for l in legs.values()), "fewer than 95% of weekday files for a leg"
cov = pd.concat({p: X.coverage_table(fr) for p, fr in frames.items()}, names=["pairing", "year"])
display(cov.round(1)); cov.to_csv(OUT / "01_coverage.csv")

## 2. Co-movement by hour (UTC)

Correlation of 5-minute returns. The expected signs: EURUSD negative with the Bund and positive with the T-Bond; GBPUSD negative with the Gilt, positive with the T-Bond; USDJPY negative with the T-Bond; EURGBP negative with the Bund, positive with the Gilt.

In [ ]:
hc = pd.concat({p: X.hourly_correlation(fr) for p, fr in frames.items()}, names=["pairing"]).reset_index(level=0)
fig, axes = plt.subplots(1, len(PAIRS), figsize=(18, 3.4), sharey=True)
for ax, p in zip(axes, PAIRS):
    h = hc[hc["pairing"] == p]
    for b in X.PAIRINGS[p]:
        ax.plot(h["hour_utc"], h[X.BOND_SHORT[b]], marker="o", ms=3, label=X.BOND_SHORT[b])
    ax.axhline(0, color="black", lw=0.5); ax.set_title(p); ax.set_xlabel("hour (UTC)"); ax.legend(fontsize=8)
axes[0].set_ylabel("corr(FX, bond) 5-min returns")
plt.tight_layout(); plt.savefig(OUT / "02_hourly_correlation.png", dpi=110); plt.show()
hc.to_csv(OUT / "02_hourly_correlation.csv", index=False)

## 3. Hedge ratios through time

Rolling 60-day no-intercept regression of FX returns on bond returns, sampled once per day.

In [ ]:
fig, axes = plt.subplots(1, len(PAIRS), figsize=(18, 3.4))
for ax, (p, fr) in zip(axes, frames.items()):
    ry, rX = fr.returns()
    beta = X.rolling_beta(ry, rX, int(60 * fr.bars_per_day))
    last = np.r_[np.flatnonzero(fr.day[1:] != fr.day[:-1]), fr.n - 1]
    for j, b in enumerate(fr.bonds):
        ax.plot(pd.to_datetime(fr.day[last]), beta[last, j], lw=0.8, label=X.BOND_SHORT[b])
    ax.axhline(0, color="black", lw=0.5); ax.set_title(f"{p}: 60-day beta"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "03_rolling_beta.png", dpi=110); plt.show()